# Notebook 03 — Experiments

Interactive version of the full experimental pipeline.

| | Experiment | Description |
|---|---|---|
| 1a | D1 random split | In-distribution baseline on UCI dataset |
| 1b | D2 random split | In-distribution baseline on PhiUSIIL dataset |
| 2a | D1 → D2 transfer | Train UCI, test on PhiUSIIL |
| 2b | D2 → D1 transfer | Train PhiUSIIL, test on UCI |

All four models are evaluated: **Random Forest**, **XGBoost**, **LightGBM**, **SVM (RBF)**.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report

from src.data_loader import load_d1, load_d2, generate_synthetic_dataset
from src.models import get_model, MODEL_NAMES, PARAM_GRIDS, ALL_MODELS
from src.evaluation import (
    compute_metrics, print_report, plot_confusion_matrix,
    plot_roc_pr_curves, plot_feature_importance,
    plot_generalisation_gap, plot_metrics_heatmap,
)
from src.features import FEATURE_NAMES

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.3})

os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../results/metrics', exist_ok=True)

print('Setup complete.')

## Configuration

Adjust `USE_REAL_DATA`, `D1_PATH`, `D2_PATH`, and `TUNE` to match your environment.

In [ ]:
D1_PATH = '../data/raw/dataset_phishing_uci.csv'
D2_PATH = '../data/raw/phiusiil_phishing_url.csv'

USE_REAL_DATA = os.path.exists(D1_PATH) and os.path.exists(D2_PATH)

# Limit dataset size for interactive exploration (set None for full run)
D1_SAMPLE = 5000    # set None for full D1
D2_SAMPLE = 10000   # set None for full D2

# Models to run — remove 'svm' if slow
MODEL_KEYS = ['rf', 'xgb', 'lgbm', 'svm']

# Hyper-parameter tuning (False = use defaults, much faster)
TUNE = False
N_ITER = 15   # RandomizedSearchCV iterations when TUNE=True

print(f'Real data available: {USE_REAL_DATA}')
print(f'Models: {MODEL_KEYS}')
print(f'Tuning: {TUNE}')

## Load Datasets

In [ ]:
if USE_REAL_DATA:
    X1, y1 = load_d1(D1_PATH, use_lexical=True, sample=D1_SAMPLE)
    X2, y2 = load_d2(D2_PATH, sample=D2_SAMPLE)
else:
    print('[FALLBACK] Using synthetic data.')
    X1, y1 = generate_synthetic_dataset(n_phishing=400, n_legit=400, random_state=42)
    X2, y2 = generate_synthetic_dataset(n_phishing=300, n_legit=300, random_state=99)

print(f'\nD1: {X1.shape}  D2: {X2.shape}')

## Helper Functions

In [ ]:
def get_proba(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    scores = model.decision_function(X)
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-10)


def tune_and_fit(model_key, X_train, y_train, tune=TUNE, n_iter=N_ITER):
    model = get_model(model_key)
    if not tune:
        model.fit(X_train, y_train)
        return model
    grid = PARAM_GRIDS.get(model_key, {})
    if not grid:
        model.fit(X_train, y_train)
        return model
    search = RandomizedSearchCV(
        model, grid, n_iter=n_iter,
        cv=StratifiedKFold(5, shuffle=True, random_state=42),
        scoring='f1', n_jobs=-1, random_state=42, verbose=0
    )
    search.fit(X_train, y_train)
    print(f'  Best CV F1: {search.best_score_:.4f}  Params: {search.best_params_}')
    return search.best_estimator_


def run_experiment(model_key, X_tr, y_tr, X_te, y_te, tag):
    model = tune_and_fit(model_key, X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = get_proba(model, X_te)
    m = compute_metrics(y_te, y_pred, y_prob)
    plot_confusion_matrix(
        y_te, y_pred,
        title=f'{MODEL_NAMES[model_key]} — {tag}',
        out_path=f'../results/figures/cm_{tag}_{model_key}.png'
    )
    plt.show()
    return model, m, y_pred, y_prob


all_results = {}
print('Helpers ready.')

---
## Experiment 1a — D1 Random Split (80/20)

In [ ]:
X1_tr, X1_te, y1_tr, y1_te = train_test_split(
    X1, y1, test_size=0.2, stratify=y1, random_state=42
)
print(f'D1 Train: {len(X1_tr):,}  Test: {len(X1_te):,}')

roc_items_1a = []
for mk in MODEL_KEYS:
    print(f'\n--- {MODEL_NAMES[mk]} ---')
    model, m, y_pred, y_prob = run_experiment(mk, X1_tr, y1_tr, X1_te, y1_te, 'exp1a_D1')
    key = f'{MODEL_NAMES[mk]} — D1 (random split)'
    all_results[key] = m
    roc_items_1a.append({'name': MODEL_NAMES[mk], 'y_true': y1_te, 'y_prob': y_prob})
    print(f'  F1={m["f1"]:.4f}  Acc={m["accuracy"]:.4f}  AUC={m["roc_auc"]:.4f}  FPR={m["false_positive_rate"]:.4f}')
    print(classification_report(y1_te, y_pred, target_names=['Legit', 'Phishing']))

In [ ]:
plot_roc_pr_curves(roc_items_1a, title_prefix='Exp1a — D1 Random Split',
                   out_path='../results/figures/roc_pr_exp1a_D1.png')
plt.show()

---
## Experiment 1b — D2 Random Split (80/20)

In [ ]:
X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2, test_size=0.2, stratify=y2, random_state=42
)
print(f'D2 Train: {len(X2_tr):,}  Test: {len(X2_te):,}')

roc_items_1b = []
for mk in MODEL_KEYS:
    print(f'\n--- {MODEL_NAMES[mk]} ---')
    model, m, y_pred, y_prob = run_experiment(mk, X2_tr, y2_tr, X2_te, y2_te, 'exp1b_D2')
    key = f'{MODEL_NAMES[mk]} — D2 (random split)'
    all_results[key] = m
    roc_items_1b.append({'name': MODEL_NAMES[mk], 'y_true': y2_te, 'y_prob': y_prob})
    print(f'  F1={m["f1"]:.4f}  Acc={m["accuracy"]:.4f}  AUC={m["roc_auc"]:.4f}  FPR={m["false_positive_rate"]:.4f}')
    print(classification_report(y2_te, y_pred, target_names=['Legit', 'Phishing']))

In [ ]:
plot_roc_pr_curves(roc_items_1b, title_prefix='Exp1b — D2 Random Split',
                   out_path='../results/figures/roc_pr_exp1b_D2.png')
plt.show()

---
## Experiment 2a — Train D1, Test D2 (generalisation)

In [ ]:
print('Cross-dataset: Train on D1 (UCI) → Test on D2 (PhiUSIIL)')
print(f'Train: {len(X1):,}  Test: {len(X2):,}')

roc_items_2a = []
for mk in MODEL_KEYS:
    print(f'\n--- {MODEL_NAMES[mk]} ---')
    model, m, y_pred, y_prob = run_experiment(mk, X1, y1, X2, y2, 'exp2a_D1toD2')
    key = f'{MODEL_NAMES[mk]} — D1→D2 (cross-dataset)'
    all_results[key] = m
    roc_items_2a.append({'name': MODEL_NAMES[mk], 'y_true': y2, 'y_prob': y_prob})
    print(f'  F1={m["f1"]:.4f}  Acc={m["accuracy"]:.4f}  AUC={m["roc_auc"]:.4f}  FPR={m["false_positive_rate"]:.4f}')
    print(classification_report(y2, y_pred, target_names=['Legit', 'Phishing']))

In [ ]:
plot_roc_pr_curves(roc_items_2a, title_prefix='Exp2a — D1→D2 Cross-dataset',
                   out_path='../results/figures/roc_pr_exp2a_D1toD2.png')
plt.show()

---
## Experiment 2b — Train D2, Test D1 (reverse transfer)

In [ ]:
print('Cross-dataset: Train on D2 (PhiUSIIL) → Test on D1 (UCI)')
print(f'Train: {len(X2):,}  Test: {len(X1):,}')

roc_items_2b = []
for mk in MODEL_KEYS:
    print(f'\n--- {MODEL_NAMES[mk]} ---')
    model, m, y_pred, y_prob = run_experiment(mk, X2, y2, X1, y1, 'exp2b_D2toD1')
    key = f'{MODEL_NAMES[mk]} — D2→D1 (cross-dataset)'
    all_results[key] = m
    roc_items_2b.append({'name': MODEL_NAMES[mk], 'y_true': y1, 'y_prob': y_prob})
    print(f'  F1={m["f1"]:.4f}  Acc={m["accuracy"]:.4f}  AUC={m["roc_auc"]:.4f}  FPR={m["false_positive_rate"]:.4f}')
    print(classification_report(y1, y_pred, target_names=['Legit', 'Phishing']))

In [ ]:
plot_roc_pr_curves(roc_items_2b, title_prefix='Exp2b — D2→D1 Cross-dataset',
                   out_path='../results/figures/roc_pr_exp2b_D2toD1.png')
plt.show()

---
## Summary — Results Table & Generalisation Gap

In [ ]:
print_report(all_results, title='ALL EXPERIMENTS — SUMMARY')

# Save metrics to JSON
with open('../results/metrics/all_metrics.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print('Saved: results/metrics/all_metrics.json')

In [ ]:
# Results as a formatted table
metrics_cols = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'false_positive_rate']
rows = []
for exp_name, m in all_results.items():
    row = {'Experiment': exp_name}
    row.update({c: round(m.get(c, float('nan')), 4) for c in metrics_cols})
    rows.append(row)
results_df = pd.DataFrame(rows).set_index('Experiment')
display(results_df.style.background_gradient(cmap='RdYlGn', subset=['f1', 'accuracy'])
                        .background_gradient(cmap='RdYlGn_r', subset=['false_positive_rate']))

In [ ]:
plot_generalisation_gap(all_results, metric='f1',
                         out_path='../results/figures/generalisation_gap_f1.png')
plt.show()

plot_metrics_heatmap(all_results,
                      out_path='../results/figures/metrics_heatmap.png')
plt.show()

## Generalisation Gap Analysis

Compute Δ-F1 = F1(random split) − F1(cross-dataset) for each model.

In [ ]:
gap_rows = []
for mk in MODEL_KEYS:
    mname = MODEL_NAMES[mk]
    f1_d1_rand  = all_results.get(f'{mname} — D1 (random split)', {}).get('f1', None)
    f1_d1_cross = all_results.get(f'{mname} — D1→D2 (cross-dataset)', {}).get('f1', None)
    f1_d2_rand  = all_results.get(f'{mname} — D2 (random split)', {}).get('f1', None)
    f1_d2_cross = all_results.get(f'{mname} — D2→D1 (cross-dataset)', {}).get('f1', None)
    gap_rows.append({
        'Model': mname,
        'D1 in-dist F1': f1_d1_rand,
        'D1→D2 F1': f1_d1_cross,
        'Δ F1 (D1)': round(f1_d1_rand - f1_d1_cross, 4) if f1_d1_rand and f1_d1_cross else None,
        'D2 in-dist F1': f1_d2_rand,
        'D2→D1 F1': f1_d2_cross,
        'Δ F1 (D2)': round(f1_d2_rand - f1_d2_cross, 4) if f1_d2_rand and f1_d2_cross else None,
    })

gap_df = pd.DataFrame(gap_rows).set_index('Model')
display(gap_df.style.background_gradient(cmap='RdYlGn_r', subset=['Δ F1 (D1)', 'Δ F1 (D2)']))